# PySEAL Implementation Example - Two

In [1]:
import pickle
import time
import random
import threading
import seal
from seal import Ciphertext
from seal import Decryptor
from seal import Encryptor
from seal import EncryptionParameters
from seal import Evaluator
from seal import IntegerEncoder
from seal import FractionalEncoder
from seal import KeyGenerator
from seal import MemoryPoolHandle
from seal import Plaintext
from seal import SEALContext
from seal import EvaluationKeys
from seal import GaloisKeys
from seal import PolyCRTBuilder
from seal import ChooserEncoder
from seal import ChooserEvaluator
from seal import ChooserPoly

### Set Encryption Parameters

- Polynomial, coefficient, plaintext modulus and then create context for encryption parameters
- Print encryption parameter information

In [2]:
parms = EncryptionParameters()
parms.set_poly_modulus("1x^2048 + 1")
parms.set_coeff_modulus(seal.coeff_modulus_128(2048))
parms.set_plain_modulus(1 << 8)
context = SEALContext(parms)

print("/ Encryption parameters:")
print("| poly_modulus: " + context.poly_modulus().to_string())
print("| coeff_modulus_size: " + (str)(context.total_coeff_modulus().significant_bit_count()) + " bits")
print("| plain_modulus: " + (str)(context.plain_modulus().value()))
print("| noise_standard_deviation: " + (str)(context.noise_standard_deviation()))

/ Encryption parameters:
| poly_modulus: 1x^2048 + 1
| coeff_modulus_size: 56 bits
| plain_modulus: 256
| noise_standard_deviation: 3.19


### Key Generation and tool setup

- Generates public/private keys and sets up tools for encryption, evaluation, and decryption

In [3]:
keygen = KeyGenerator(context)
public_key = keygen.public_key()
secret_key = keygen.secret_key()
encryptor = Encryptor(context, public_key)
evaluator = Evaluator(context)
decryptor = Decryptor(context, secret_key)

### Data setup

- Lists values and weights, and sets up an encoder for fractional (real number) encoding

In [4]:
rational_numbers = [3.1, 4.159, 2.65, 3.5897, 9.3, 2.3, 8.46, 2.64, 3.383, 2.795]
coefficients = [0.1, 0.05, 0.05, 0.2, 0.05, 0.3, 0.1, 0.025, 0.075, 0.05]

encoder = FractionalEncoder(context.plain_modulus(), context.poly_modulus(), 64, 32, 3)

### Encryption

- Encrypt each rational number and encode each coefficient (weights remain plaintext)
- Encodes and enecrypts all rational numbers

In [5]:
encrypted_rationals = []
rational_numbers_string = "Encoding and encrypting: "
encoded_coefficients = []
encoded_coefficients_string = "Encoding plaintext coefficients: "
for i in range(10):
    encrypted_rationals.append(Ciphertext(parms))
    encryptor.encrypt(encoder.encode(rational_numbers[i]), encrypted_rationals[i])
    rational_numbers_string += (str)(rational_numbers[i])[:6]
    if i < 9: 
        rational_numbers_string += ", "
    
    encoded_coefficients.append(encoder.encode(coefficients[i]))
    encoded_coefficients_string += (str)(coefficients[i])[:6]
    if i < 9: 
        encoded_coefficients_string += ", "
        
print(rational_numbers_string)
print(encoded_coefficients_string)

Encoding and encrypting: 3.1, 4.159, 2.65, 3.5897, 9.3, 2.3, 8.46, 2.64, 3.383, 2.795
Encoding plaintext coefficients: 0.1, 0.05, 0.05, 0.2, 0.05, 0.3, 0.1, 0.025, 0.075, 0.05


### Perform Homomorphic Evaluations

- Performs multiplications of ciphertexts and plaintexts, resulting in encrypted weighted terms.
- Computes the sum of encrypted weighted values, producing an encrypted total
- Homomorphically scales down the sum
- Prints how much noise margin remains, important for managing encryption depth

In [6]:
div_by_ten = encoder.encode(0.1)

print("Computing products: ")
for i in range(10):
    evaluator.multiply_plain(encrypted_rationals[i], encoded_coefficients[i])
print("Done")

encrypted_result = Ciphertext()
print("Adding up all 10 ciphertexts: ")
evaluator.add_many(encrypted_rationals, encrypted_result)
print("Done")

print("Dividing by 10: ")
evaluator.multiply_plain(encrypted_result, div_by_ten)
print("Done")

print("Noise budget in result: " + (str)(decryptor.invariant_noise_budget(encrypted_result)) + " bits")

Computing products: 
Done
Adding up all 10 ciphertexts: 
Done
Dividing by 10: 
Done
Noise budget in result: 31 bits


### Decrypt and decode final result

- Decrypts and decodes the weighted average result and prints it

In [7]:
plain_result = Plaintext()
print("Decrypting result: ")
decryptor.decrypt(encrypted_result, plain_result)
print("Done")
result = encoder.decode(plain_result)
print("Weighted average: " + (str)(result)[:8])
print("THIS IS THE PLAIN RESULT: ", result)

Decrypting result: 
Done
Weighted average: 0.382886
THIS IS THE PLAIN RESULT:  0.3828864999999884
